# 다음 카페 게시판 게시글 제목 크롤링
- 대상 게시판: https://cafe.daum.net/skfootball/IxV3
- 다음 카페는 iframe 구조라 직접 `_c21_` 내부 URL을 사용해야 함

In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

ModuleNotFoundError: No module named 'bs4'

In [2]:
# 다음 카페 내부 URL 패턴으로 게시판 목록 접근
# grpid=1O7ju (skfootball 카페), fldid=IxV3 (해당 게시판)
url = "https://cafe.daum.net/_c21_/bbs_list?grpid=1O7ju&fldid=IxV3"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Referer": "https://cafe.daum.net/skfootball/IxV3"
}

response = requests.get(url, headers=headers)
print("Status:", response.status_code)
print("Content length:", len(response.text))

Status: 200
Content length: 76486


In [3]:
# HTML 파싱
soup = BeautifulSoup(response.text, "html.parser")

# 페이지 구조 확인
print(soup.prettify()[:3000])

<!DOCTYPE html>
<html class="" lang="ko">
 <head>
  <meta content="width=device-width, initial-scale=1" name="viewport"/>
  <meta content="text/html; charset=utf-8" http-equiv="content-type"/>
  <meta content="ie=edge" http-equiv="X-UA-Compatible"/>
  <meta content="서경 축구클럽 총모임 - [조기축구동호회친선경기]" property="og:site_name"/>
  <meta content="_서울북부 평일축구" property="og:title"/>
  <meta content="서경 축구클럽 총모임 - [조기축구동호회친선경기]" property="og:article:author"/>
  <meta content="글 작성시 말머리를 달아주세요~!" property="og:description"/>
  <meta content="https://t1.daumcdn.net/cafe_image/cafe_meta_image_240307.png" property="og:image"/>
  <meta content="모든 이야기의 시작, Daum 카페" name="description"/>
  <meta content="noindex,indexifembedded" name="robots"/>
  <meta content="unsafe-url" name="referrer"/>
  <script src="//t1.daumcdn.net/cafe_cj/pcweb/js/1/jquery-1.12.4.min.js">
  </script>
  <script src="//t1.daumcdn.net/cssjs/userAgent/userAgent-1.0.14.min.js" type="text/javascript">
  </script>
  <title>
   서경 축구클럽 총모임 

In [4]:
# 게시글 제목 추출
# 다음 카페 게시판 목록의 제목 태그 선택자
titles = []

# 방법 1: .article-item 클래스
items = soup.select(".article-item")
print(f"article-item 개수: {len(items)}")

for item in items:
    title_tag = item.select_one(".tit-box")
    if title_tag:
        titles.append(title_tag.get_text(strip=True))

print(f"추출된 제목 수: {len(titles)}")
for t in titles[:10]:
    print(t)

article-item 개수: 0
추출된 제목 수: 0


In [5]:
# 방법 2: 다른 선택자 시도 (방법 1이 안 될 경우)
# 링크 태그에서 게시글 관련 것만 추출
all_links = soup.find_all("a")
print(f"전체 링크 수: {len(all_links)}")

# 게시글 링크 필터 (datanum 파라미터 포함)
post_links = [a for a in all_links if a.get("href") and "datanum" in str(a.get("href", ""))]
print(f"게시글 링크 수: {len(post_links)}")

for link in post_links[:10]:
    print(link.get_text(strip=True), "|", link.get("href"))

전체 링크 수: 27
게시글 링크 수: 0


In [6]:
# 방법 3: tbody 내 tr 태그 (테이블 형식 게시판)
rows = soup.select("tbody tr")
print(f"tr 행 수: {len(rows)}")

for row in rows[:10]:
    print(row.get_text(strip=True)[:100])

tr 행 수: 0


## 디버깅: 실제로 뭐가 HTML에 들어있나 확인

In [7]:
# 27개 링크 전체 출력
for a in soup.find_all("a"):
    print(repr(a.get_text(strip=True)[:50]), "|", a.get("href", "")[:100])

'다음 카페의 ie10 이하 브라우저 지원이 종료됩니다. 원활한 카페 이용을 위해 사용 중인' | https://cafe.daum.net/supporters/MbmU/150
'Daum' | https://www.daum.net/?t__nil_navi=daum
'카페' | https://top.cafe.daum.net?t__nil_navi=cafehome
'테이블' | https://table.cafe.daum.net
'메일' | https://mail.daum.net?t__nil_navi=mailhome
'즐겨찾는 카페' | #
'로그인' | #
'최신글 보기' | /_c21_/recent_bbs_list?grpid=1O7ju&fldid=_rec
'인기글 보기' | /_c21_/favor_bbs_list?grpid=1O7ju
'공지사항' | /_c21_/bbs_list?grpid=1O7ju&fldid=IxUh
'우리팀 소개' | /_c21_/bbs_list?grpid=1O7ju&fldid=IxUg
'용병 신청&구함' | /_c21_/bbs_list?grpid=1O7ju&fldid=IxUe
'플래티넘 (공개)' | /_c21_/cafe_profile?grpid=1O7ju
'공차야산다' | #
'회원수' | /_c21_/memberlist?grpid=1O7ju
'84,575' | /_c21_/memberlist?grpid=1O7ju
'카페앱수' | https://cafe-notice.tistory.com/2328
'' | /_c21_/join_register?grpid=1O7ju
'검색' | #
'카페 전체 메뉴' | #
'▲' | #
'서비스 약관/정책' | https://policy.daum.net/policy/info
'권리침해신고' | https://cs.daum.net/redbell/top.html
'이용약관' | https://top.cafe.daum.net/_c21_/agreement_axz
'카페 고객센터' | https://cs.daum.net/f

In [8]:
import re

raw = response.text

# raw HTML에 datanum 있는지 확인
print("datanum in raw HTML:", "datanum" in raw)

datanums = re.findall(r'datanum[=:]+(\d+)', raw)
print("발견된 datanum들:", datanums[:20])

# bbs_read 링크 패턴
bbs_reads = re.findall(r'bbs_read[^"\'\\s]*', raw)
print("\nbbs_read 패턴들:", bbs_reads[:10])

datanum in raw HTML: False
발견된 datanum들: []

bbs_read 패턴들: ['bbs_read a.btn:fir']


## bbs_read 방식으로 전환
crawling_trial1에서 확인: `_c21_/bbs_read` URL은 requests만으로 실제 HTML 반환됨  
→ 게시글 상세 페이지 하단에 같은 게시판 글 목록이 포함되어 있어서 datanum 목록 수집 가능

In [11]:
# bbs_list 에서 datanum을 못 찾으면 bbs_read로 IxV3 게시판 아무 글이나 찾기
# bbs_list raw HTML에서 IxV3 관련 datanum 추출 시도
if datanums:
    start_datanum = datanums[0]
    print(f"bbs_list에서 찾은 datanum: {start_datanum}")
else:
    # bbs_list 페이지의 og:title 이 "_서울북부 평일축구" 처럼 게시글 제목이 있으므로
    # 해당 게시글이 최근 글임 → datanum을 모르니 bbs_list HTML 더 탐색
    print("datanum 없음, HTML 끝부분 확인")
    print(raw[-3000:])

datanum 없음, HTML 끝부분 확인
44\uB9AC\uC218 \uB9AC\uADF8(\uD3D0\uC1C4\uC608\uC815)',
		})
							boards.push({
			fldid: 'TJDk',
			fldname: '\u25A0\u25A0 \uC2DC\/\uB3C4\/\uAD6C\/\uAD70 \uD611\uD68C\uB300\uD68C(\uD3D0\uC1C4\uC608\uC815)',
		})
							boards.push({
			fldid: 'U9qk',
			fldname: '\u25A0\u25A0 \uC720\uC18C\uB144 \uB300\uD68C&\uAD50\uB958\uC804(\uD3D0\uC1C4\uC608\uC815)',
		})
							boards.push({
			fldid: 'UPDv',
			fldname: '\u25A0\u25A0 2016 \uC81C2\uD68C \uAC8C\uD1A0\uB808\uC774 GUFA CUP(\uD3D0\uC1C4\uC608\uC815)',
		})
							boards.push({
			fldid: 'Sp61',
			fldname: '\u25A1\u25A1 2015 \uC11C\uC6B8 \uC2A4\uB9C8\uC77C \uB9AC\uADF8(\uD3D0\uC1C4\uC608\uC815)',
		})
							boards.push({
			fldid: 'TNqN',
			fldname: '\u25A1\u25A1 2014 \uB4DC\uB9BC\uCEF5 \uB9AC\uADF8(\uD3D0\uC1C4\uC608\uC815)',
		})
							boards.push({
			fldid: 'Uc84',
			fldname: '_VENTUS',
		})
							boards.push({
			fldid: 'UcNt',
			fldname: '_\uBCA4\uD22C\uC2A4SC',
		})
			</script>

<link rel="style

In [13]:
# start_datanum을 위 셀 결과로 채우거나, 직접 지정
# (브라우저로 https://cafe.daum.net/skfootball/IxV3 열어서 아무 글 URL의 숫자 확인)
start_datanum = 136431  # 또는 직접 입력: "12345"

test_url = f"https://cafe.daum.net/_c21_/bbs_read?grpid=1O7ju&fldid=IxV3&datanum={start_datanum}"
print("접속 URL:", test_url)

res2 = requests.get(test_url, headers=headers)
print("Status:", res2.status_code, "| 크기:", len(res2.text))

soup2 = BeautifulSoup(res2.text, "html.parser")

og_title = soup2.find("meta", {"property": "og:title"})
print("og:title:", og_title["content"] if og_title else "없음")

접속 URL: https://cafe.daum.net/_c21_/bbs_read?grpid=1O7ju&fldid=IxV3&datanum=136431
Status: 404 | 크기: 4657
og:title: 없음


In [14]:
# bbs_read 페이지에서 같은 게시판의 글 목록 datanum 전체 수집
raw2 = res2.text

all_datanums_in_page = list(set(re.findall(r'datanum[=:]+(\d+)', raw2)))
print(f"페이지 내 datanum {len(all_datanums_in_page)}개:", all_datanums_in_page[:20])

# bbs_read 링크 패턴으로 글 목록 링크 추출
bbs_read_in_page = re.findall(r'bbs_read\?[^"\'<\s]+', raw2)
print(f"\nbbs_read 링크 {len(bbs_read_in_page)}개:")
for link in bbs_read_in_page[:20]:
    print(" ", link)

페이지 내 datanum 0개: []

bbs_read 링크 0개:


## HTML에 articles JS 변수가 있는지 확인
bbs_list 페이지 하단 config에 `articles: articles`가 있음 → JS 변수로 미리 주입돼 있을 가능성

In [15]:
# articles.push 패턴 탐색
articles_pushes = re.findall(r'articles\.push\(\{[^}]+\}\)', raw)
print(f"articles.push 개수: {len(articles_pushes)}")
for a in articles_pushes[:3]:
    print(a[:300])
    print()

# var articles 선언 탐색
idx = raw.find("var articles")
if idx != -1:
    print("var articles 발견:")
    print(raw[idx:idx+500])
else:
    print("var articles 없음")

articles.push 개수: 20
articles.push({
		dataid: '24673',
		grpid: '1O7ju',
		fldid: 'IxV3',
		bbsdepth: '006Pxzzzzzzzzzzzzzzzzzzzzzzzzz',

		title: '2026\uB144 5\uC6D4 25\uC77C \uC6D4\uC694\uC77C \uCD08\uC548\uC0B0 \uAD6C\uC7A5 20:00~22:00 \uACBD\uAE30 \uCD08\uCCAD\uB4DC\uB9BD\uB2C8\uB2E4.',
		headCont: '',
		board: '',


articles.push({
		dataid: '24672',
		grpid: '1O7ju',
		fldid: 'IxV3',
		bbsdepth: '006Pwzzzzzzzzzzzzzzzzzzzzzzzzz',

		title: '(\uAC15\uC11C\uAD6C) 6\/30 (\uD654) \uAC00\uC591\uAD6C\uC7A5 20-22\uC2DC \uCD08\uCCAD\uD569\uB2C8\uB2E4.',
		headCont: '',
		board: '',
		vldstatus: '',
		commentCnt: 0,
		i

articles.push({
		dataid: '24670',
		grpid: '1O7ju',
		fldid: 'IxV3',
		bbsdepth: '006Puzzzzzzzzzzzzzzzzzzzzzzzzz',

		title: '2026\uB144 5\uC6D4 25\uC77C \uC6D4\uC694\uC77C \uCD08\uC548\uC0B0 \uAD6C\uC7A5 20:00~22:00 \uACBD\uAE30 \uCD08\uCCAD\uB4DC\uB9BD\uB2C8\uB2E4.',
		headCont: '',
		board: '',


var articles 발견:
var articles = [];
		articles.push({
		dataid: '24673',


In [17]:
import json

titles_from_js = []
for item in articles_pushes:
    dataid_m = re.search(r"dataid:\s*'(\d+)'", item)
    title_m  = re.search(r"title:\s*'([^']*)'", item)
    if dataid_m and title_m:
        raw_title = title_m.group(1)
        # JS 유니코드 이스케이프 디코딩 (년 → 년)
        decoded_title = json.loads(f'"{raw_title}"')
        titles_from_js.append({
            "dataid": dataid_m.group(1),
            "title": decoded_title
        })

print(f"추출된 제목 {len(titles_from_js)}개:")
for t in titles_from_js:
    print(t["dataid"], "|", t["title"])

추출된 제목 20개:
24673 | 2026년 5월 25일 월요일 초안산 구장 20:00~22:00 경기 초청드립니다.
24672 | (강서구) 6/30 (화) 가양구장 20-22시 초청합니다.
24670 | 2026년 5월 25일 월요일 초안산 구장 20:00~22:00 경기 초청드립니다.
24669 | [마감/마감]5월 21일 목요일 21~23시 용마폭포공원축구장 상대 팀 모십니다.
24668 | [노원구] 5월 27일(수) 20시 초청합니다.
24667 | 05월 25일 월요일 21시~23시 용마폭포공원축구장 초청합니다.
24665 | 5.25 월 오후 5-7 제2월곡 초청
24663 | 26.05.25(월, 대체공휴일) 새벽 06-08시 의정부 경민대학교 인조잔디구장 11:11 무료초청
24662 | 5월26일(화) 21시 남양주크낙새구장(2월잔디교체) 무료초청합니다.
24661 | 5월22일(금)/21-23시/국민대학교 대운동장/ 축구 초청합니다
24659 | 6월 평일 저녁20-22시 초청합니다.
24658 | 2026년 5월 25일 월요일 초안산 구장 20:00~22:00 경기 초청드립니다.
24657 | 5월 21일 목요일 20-22 노원구 초청합니다.
24656 | 6월 평일 월요일 저녁 8시 노원구 축구 초청 및 초청 요청 드립니다.
24655 | 5월 25일 평일 월요일 저녁 8시 불암산 축구장 초청드립니다.
24654 | 5월 27일(수) 오후 7-9시 응봉체육공원(최근 리모델링) 초청합니다.
24653 | 5월22일 금요일 19:00~21:00 용마폭포공원에서 매칭 구합니다!
24652 | 21일목요일 별내에코랜드 20~23시 초청합니다
24651 | 5월27일 수요일 강북구민운동장 20-22
24650 | 5월 25일(월) 노원구 초안산 구장 20~22시 초청드립니다.


## 전체 페이지 수집 (1페이지 20개 × 833페이지)
`page=N` 파라미터로 다음 페이지 접근 가능

In [ ]:
import time

def get_titles_by_page(page_num):
    url = f"https://cafe.daum.net/_c21_/bbs_list?grpid=1O7ju&fldid=IxV3&page={page_num}"
    res = requests.get(url, headers=headers)
    raw = res.text

    pushes = re.findall(r'articles\.push\(\{[^}]+\}\)', raw)
    titles = []
    for item in pushes:
        dataid_m = re.search(r"dataid:\s*'(\d+)'", item)
        title_m  = re.search(r"title:\s*'([^']*)'", item)
        if dataid_m and title_m:
            decoded = json.loads(f'"{title_m.group(1)}"')
            titles.append({"dataid": dataid_m.group(1), "title": decoded})

    return titles


# 3페이지 수집 테스트
all_titles = []

for page in range(1, 4):
    titles = get_titles_by_page(page)
    print(f"--- page {page} ({len(titles)}개) ---")
    for t in titles[:3]:
        print(f"  {t['dataid']} | {t['title']}")
    all_titles.extend(titles)
    time.sleep(1)

print(f"\n총 수집: {len(all_titles)}개")


: 

## 1페이지 제목 파싱 — 날짜 / 지역 / 시간 / 초청 구분

In [21]:
def parse_title(title):
    result = {"title": title, "날짜": None, "지역": None, "장소": None, "시간": None, "초청구분": None}

    # ── 날짜 ──────────────────────────────────────────────────────
    date_patterns = [
        r'\d{2}\.\d{2}\.\d{2}',          # 26.05.25
        r'\d{1,2}월\s*\d{1,2}일',         # 5월 25일
        r'\d{1,2}\.\d{1,2}\b',            # 5.25
        r'\d{1,2}\/\d{1,2}',              # 6/30
    ]
    for pat in date_patterns:
        m = re.search(pat, title)
        if m:
            result["날짜"] = m.group(0)
            break

    # ── 시간 ──────────────────────────────────────────────────────
    time_patterns = [
        r'(?:오전|오후|새벽|저녁)?\s*\d{1,2}:\d{2}\s*[~\-]\s*\d{1,2}:\d{2}',  # 20:00~22:00
        r'(?:오전|오후|새벽|저녁)?\s*\d{1,2}\s*[시~\-]\s*\d{1,2}\s*시?',        # 20~22시
        r'(?:오전|오후|새벽|저녁)\s*\d{1,2}',                                    # 오후 8
    ]
    for pat in time_patterns:
        m = re.search(pat, title)
        if m:
            result["시간"] = m.group(0).strip()
            break

    # ── 지역 (행정구역: 구/시/군) — 한글만, 숫자 제외 ────────────────
    bracket_m = re.search(r'[\[\(]([가-힣]{2,6}[구시군])[\]\)]', title)
    if bracket_m:
        result["지역"] = bracket_m.group(1)
    else:
        area_m = re.search(r'([가-힣]{2,5}[구시군])[\s\W]', title)
        if area_m:
            result["지역"] = area_m.group(1)

    # ── 장소 (경기장/구장/운동장 등) ─────────────────────────────────
    venue_m = re.search(
        r'[가-힣a-zA-Z0-9]+(?:구장|운동장|체육공원|체육관|인조잔디구장|잔디구장|대운동장|에코랜드|공원)',
        title
    )
    if venue_m:
        result["장소"] = venue_m.group(0)

    # ── 초청 구분 ──────────────────────────────────────────────────
    if re.search(r'초청\s*(요청|해주|구함|구합)', title):
        result["초청구분"] = "요청"
    elif re.search(r'모십니다|구합니다|구해요|찾습니다|매칭\s*구', title):
        result["초청구분"] = "요청"
    elif re.search(r'초청|무료초청', title):
        result["초청구분"] = "초청"
    else:
        result["초청구분"] = "기타"

    return result


parsed = [parse_title(t["title"]) for t in titles_from_js]
df = pd.DataFrame(parsed)
df

,title,날짜,지역,장소,시간,초청구분
0,2026년 5월 25일 월요일 초안산 구장 20:00~22:00 경기 초청드립니다.,5월 25일,None,None,20:00~22:00,초청
1,(강서구) 6/30 (화) 가양구장 20-22시 초청합니다.,6/30,강서구,가양구장,20-22시,초청
2,2026년 5월 25일 월요일 초안산 구장 20:00~22:00 경기 초청드립니다.,5월 25일,None,None,20:00~22:00,초청
3,[마감/마감]5월 21일 목요일 21~23시 용마폭포공원축구장 상대 팀 모십니다.,5월 21일,None,용마폭포공원축구장,21~23시,요청
4,[노원구] 5월 27일(수) 20시 초청합니다.,5월 27일,노원구,None,None,초청
5,05월 25일 월요일 21시~23시 용마폭포공원축구장 초청합니다.,05월 25일,None,용마폭포공원축구장,None,초청
6,5.25 월 오후 5-7 제2월곡 초청,5.25,None,None,오후 5-7,초청
7,"26.05.25(월, 대체공휴일) 새벽 06-08시 의정부 경민대학교 인조잔디구장 ...",26.05.25,None,인조잔디구장,새벽 06-08시,초청
8,5월26일(화) 21시 남양주크낙새구장(2월잔디교체) 무료초청합니다.,5월26일,None,남양주크낙새구장,None,초청
9,5월22일(금)/21-23시/국민대학교 대운동장/ 축구 초청합니다,5월22일,None,대운동장,21-23시,초청
